# Phase 5 experiment B -- does a second (coronal) series rescue the coronal labels?

**One variable changes from Phase 4 run 1: `MAX_SERIES` 1 -> 2.** Same frozen `primary_v2` folds,
same gold exclusion, same pseudo-labels, same `efficientnet_b0`, same Adam 1e-4 / batch 8 / 1 epoch.
So the paired OOF delta is attributable to the extra series and nothing else.

**The hypothesis comes from run 1's own failure pattern.** Gold transfer put MCL at 0.5034 --
literally random -- and Lateral OA at 0.6132, while Effusion reached 0.9391. MCL and the OA
compartments are *coronal* findings; run 1 fed the model a single **sagittal** series, so it could
not see them. The Qwen3-4B label for MCL scored 0.887 gold AUC in the bake-off, so the label is
fine and the input is the constraint. Next in `_SERIES_PRIORITY` after sagittal-fluid-sensitive is
**coronal**-fluid-sensitive, which is exactly what `max_series=2` adds.

**Prediction, written before the run:** MCL and Lateral OA move most; Effusion and ACL (sagittal
findings, already working) barely move. If instead everything rises uniformly, the gain is extra
data rather than the coronal plane, and the follow-up is slice count, not series routing.

Cost: 2x run 1's forward volume, ~13 min/fold, ~70 min for 5 folds.

Two correctness fixes carried in from reviewing run 1:
- **The fold assignment is asserted by digest**, not by fold sizes. Run 1's
  `_counts == [881,881,881,882,882]` passes for *any* seed -- it checks the distribution, not the
  assignment -- and every paired delta from here on depends on `primary_v2` being the same 4,407
  study-to-fold mapping it was in run 1.
- **The gold tier reloads checkpoints with `pretrained=False`.** Run 1 passed `pretrained=True`
  there and then overwrote every weight; harmless with internet on, a hang with it off.


In [ ]:
import glob, hashlib, os, shutil, sys, time

GIT_SHA = 'PENDING'
# sha256 over 'uid,fold' lines of the sorted assignment in results/folds_primary_v2.csv
FOLDS_PRIMARY_V2_SHA256 = 'cace8c45733ee7920a442fa4ae3f1db4a0d76401504e4729b49562d5eab52047'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
# prepped artifacts live in the four Phase 2 shard kernels' outputs
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
assert len(PREPPED_DIRS) == 4, f'expected 4 prep-shard outputs, found {len(PREPPED_DIRS)}'

uid_to_npz = {}
for d in PREPPED_DIRS:
    for p in sorted(glob.glob(os.path.join(d, '*.npz'))):
        uid = os.path.splitext(os.path.basename(p))[0]
        assert uid not in uid_to_npz, f'duplicate artifact for {uid}'
        uid_to_npz[uid] = p
print(f'{len(uid_to_npz)} prepped artifacts mounted')

# The four shard outputs live in separate directories, but PreppedStudyDataset
# takes a single npz_root -- so symlink every artifact into one flat working
# directory rather than touching the tested loader. Symlinks cost ~nothing and
# the dataset reads straight through them.
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)
print('symlink farm ready at', NPZ_ROOT)

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print('src/knee mounted from', SRC)


In [ ]:
import torch

assert torch.cuda.is_available(), 'no GPU attached -- pick GPU T4 x2 before Save & Run All'
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, sm_{major}{minor}')
    # CLI pushes land on P100 essentially every time; fail fast here rather
    # than deep inside the first batch-norm kernel launch.
    assert (major, minor) >= (7, 0), (
        f'GPU {i} is sm_{major}{minor} (P100?) -- stop, select GPU T4 x2 in the editor '
        'accelerator settings, then Save Version -> Save & Run All (Commit)')
device = 'cuda'


In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.train import (
    Timer,
    evaluate,
    load_gold_holdout,
    log_experiment,
    make_folds,
    train_one_epoch,
    train_val_split,
)
from knee.dataset import PreppedStudyDataset
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.model import KneeModel

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

# --- pseudo-labels: score_* renamed to the bare LABEL_COLUMNS the dataset
# contract expects; weight_* kept aside for later experiment rows ---
pseudo_path = glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0]
pseudo = pd.read_csv(pseudo_path)
weights_all = pseudo[['StudyInstanceUID'] + [f'weight_{l}' for l in LABEL_COLUMNS]].copy()
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]].copy()
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS
weights_all.columns = ['StudyInstanceUID'] + [f'w_{l}' for l in LABEL_COLUMNS]
assert len(labels_all) == 4407 and labels_all['StudyInstanceUID'].is_unique
_vals = labels_all[LABEL_COLUMNS].to_numpy(dtype=float)
assert np.isfinite(_vals).all() and (_vals >= 0).all() and (_vals <= 1).all()
labels_all['StudyInstanceUID'] = labels_all['StudyInstanceUID'].astype(str)

# Evaluation tier uses the pseudo-labels BINARIZED at 0.5: sklearn's AUC
# requires binary targets, and ranking against the LLM's discretized judgment
# is the interpretable quantity. Training itself stays on the soft scores --
# that is where their calibration carries information.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (_vals >= 0.5).astype(float)
print('binarized positive rates:',
      {l: round(float(labels_eval[l].mean()), 4) for l in LABEL_COLUMNS})

# --- frozen fold assignment: make_folds is deterministic and order-independent,
# so re-deriving it here reproduces folds_primary_v2 exactly (verified locally) ---
folds = make_folds(all_uids, n_folds=5, seed=0)
# Digest, not fold sizes: [881,881,881,882,882] holds for any seed, so run 1's
# size check would not have noticed primary_v2 silently changing underneath a
# paired comparison. This pins the exact study -> fold mapping, and matches
# results/folds_primary_v2.csv (verified locally).
_digest = hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()).hexdigest()
assert _digest == FOLDS_PRIMARY_V2_SHA256, (
    f'fold assignment is not primary_v2 ({_digest}) -- every paired delta '
    'against run 1 would be meaningless')
print('fold assignment verified against primary_v2')

# --- gold holdout via the one tested exclusion path; excluded from BOTH sides
# of every split, they become the post-training transfer tier ---
gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58
gold_df[['StudyInstanceUID']].to_csv('gold_tmp.csv', index=False)
holdout = load_gold_holdout('gold_tmp.csv')
assert len(holdout) == 58

splits = {}
for vf in range(5):
    tr, va = train_val_split(folds, val_fold=vf, exclude_uids=holdout)
    assert not set(tr) & holdout and not set(va) & holdout
    # per fold: train+val covers every non-gold study exactly once
    assert set(tr) | set(va) == set(all_uids) - holdout
    splits[vf] = (tr, va)
print('split sizes (train/val):',
      [(len(t), len(v)) for t, v in splits.values()])
print(f'gold holdout: {len(holdout)} studies excluded from every split')

# --- every training study must have an artifact on disk ---
missing_artifacts = [u for t, v in splits.values() for u in t + v if u not in uid_to_npz]
assert not missing_artifacts, f'{len(missing_artifacts)} studies lack prepped artifacts'
print('artifact coverage: complete')


In [ ]:
N_FOLDS, SEED = 5, 0
EPOCHS_PER_FOLD = 1
BATCH_SIZE = 8
MAX_SERIES, N_SLICES = 2, 16   # run 1 was 1 x 16 -- the second series is the experiment
CONFIG_HASH = f'phase5b_pseudo_qwen3_4b_efficientnet_b0_s{MAX_SERIES}x{N_SLICES}_e{EPOCHS_PER_FOLD}_cuda'
FOLD_SET = 'primary_v2'
EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_REAL_HEADER = (
    'date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
    'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
    'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
    'macro_auc,paired_delta,train_minutes,inference_seconds,promoted'
)
with open(EXPERIMENTS_CSV, 'w') as f:
    f.write(_REAL_HEADER + '\n')

HYPOTHESIS = ('Phase 5 exp B: 2 series x 16 slices (sagittal + coronal) vs run 1s 1 x 16; '
              'coronal labels (MCL 0.503, Lateral OA 0.613 on gold) should move most')

labeled_uids = [u for u in all_uids if u not in holdout]
oof_true = np.full((len(labeled_uids), len(LABEL_COLUMNS)), np.nan)
oof_pred = np.full_like(oof_true, np.nan)
row_of = {u: i for i, u in enumerate(labeled_uids)}

from tqdm.auto import tqdm

for fold in range(N_FOLDS):
    train_uids, val_uids = splits[fold]
    train_ds = PreppedStudyDataset(train_uids, NPZ_ROOT, labels_df=labels_all,
                                   n_slices=N_SLICES, max_series=MAX_SERIES)
    # val carries BINARIZED targets: evaluate() returns them as y_true and the
    # per-label AUCs need binary targets (see note where labels_eval is built)
    val_ds = PreppedStudyDataset(val_uids, NPZ_ROOT, labels_df=labels_eval,
                                 n_slices=N_SLICES, max_series=MAX_SERIES)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_timer, infer_timer = Timer(), Timer()
    with train_timer:
        for epoch in range(EPOCHS_PER_FOLD):
            loss = train_one_epoch(model, tqdm(
                train_loader, desc=f'fold {fold} epoch {epoch}', leave=False),
                optimizer, device=device)
    with infer_timer:
        yt, yp = evaluate(model, val_loader, device=device)

    for i, u in enumerate(val_uids):
        r = row_of[u]
        oof_true[r], oof_pred[r] = yt[i], yp[i]

    aucs = dict(zip(LABEL_COLUMNS, per_label_auc(yt, yp)))
    macro = float(np.nanmean(list(aucs.values())))
    log_experiment(
        EXPERIMENTS_CSV,
        git_sha=GIT_SHA, config_hash=CONFIG_HASH, hypothesis=HYPOTHESIS,
        fold_set=FOLD_SET, seed=SEED,
        per_label_auc=aucs, macro_auc=macro, paired_delta=0.0,
        train_minutes=train_timer.minutes, inference_seconds=infer_timer.total_seconds,
        promoted=False,
    )
    torch.save(model.state_dict(), f'/kaggle/working/knee_phase5b_fold{fold}.pt')
    print(f'fold {fold}: macro {macro:.4f} | train {train_timer.minutes:.1f}min | '
          f'infer {infer_timer.total_seconds:.1f}s', flush=True)
    del model
    import gc; gc.collect(); torch.cuda.empty_cache()

pooled_macro = macro_auc(oof_true, oof_pred)
print(f'\npooled OOF macro AUC vs pseudo-labels ({len(labeled_uids)} studies): '
      f'{pooled_macro:.4f}')
pl = dict(zip(LABEL_COLUMNS, per_label_auc(oof_true, oof_pred)))
for k, v in pl.items():
    print(f'  {k:20s} {v:.4f}')
np.save('/kaggle/working/oof_true.npy', oof_true)
np.save('/kaggle/working/oof_pred.npy', oof_pred)
np.save('/kaggle/working/oof_uids.npy', np.array(labeled_uids))

# --- the paired delta the promotion rule actually turns on -------------------
# Run 1's OOF arrays ride in on the knee-phase4-train kernel output: same 4,349
# studies, same frozen folds, same binarized targets, so studies pair one to one
# and the bootstrap resamples both arms together. Degrades to 0.0 rather than
# failing if that input was not attached -- the delta can also be computed
# locally from the two kernels' outputs.
# NaN, not 0.0: 0.0 is a real result ("no difference"), and the run 1 rows
# already carry a legitimate 0.0. An uncomputed delta has to be distinguishable
# from a measured one or the column stops meaning anything.
paired = float('nan')
_run1 = glob.glob('/kaggle/input/**/oof_pred.npy', recursive=True)
if _run1:
    base_pred = np.load(_run1[0])
    base_true = np.load(glob.glob('/kaggle/input/**/oof_true.npy', recursive=True)[0])
    base_uids = np.load(glob.glob('/kaggle/input/**/oof_uids.npy', recursive=True)[0])
    assert list(base_uids) == labeled_uids, 'run 1 OOF covers different studies -- not pairable'
    assert np.array_equal(base_true, oof_true), 'run 1 scored against different targets'
    paired, lo, hi = paired_macro_auc_delta(oof_true, base_pred, oof_pred, n_boot=1000, seed=0)
    verdict = 'clears noise' if (lo > 0 or hi < 0) else 'does NOT clear noise'
    print(f'\npaired macro-AUC delta vs run 1: {paired:+.4f}  '
          f'95% CI [{lo:+.4f}, {hi:+.4f}] -- {verdict}')
    print('per-label delta (exp B - run 1):')
    for k, a, b in zip(LABEL_COLUMNS, per_label_auc(oof_true, base_pred),
                       per_label_auc(oof_true, oof_pred)):
        print(f'  {k:20s} {b - a:+.4f}')
else:
    print('\nrun 1 OOF arrays not attached -- paired delta logged as NaN; '
          'compute it locally from the two kernels\' oof_pred.npy with '
          'knee.metrics.paired_macro_auc_delta')

# One pooled summary row carrying the number the promotion rule reads. The
# per-fold rows above cannot hold it: a paired delta is a property of the whole
# OOF, not of one fold.
log_experiment(
    EXPERIMENTS_CSV,
    git_sha=GIT_SHA, config_hash=CONFIG_HASH, hypothesis=HYPOTHESIS,
    fold_set='primary_v2_pooled', seed=SEED,
    per_label_auc=pl, macro_auc=pooled_macro, paired_delta=paired,
    train_minutes=0.0, inference_seconds=0.0, promoted=False,
)


In [ ]:
import gc

# --- gold transfer tier --------------------------------------------------
# Every fold model scores the 58 held-out gold studies out-of-training; their
# TRUE rubric labels come straight from train.csv. This measures
# LLM-pseudo-label -> rubric transfer directly -- the number Phase 1 could
# only back out of the LB (~0.674 on 4 labels).
gold_labels_df = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_uids = gold_labels_df['StudyInstanceUID'].astype(str).tolist()
gold_ds = PreppedStudyDataset(gold_uids, NPZ_ROOT, labels_df=gold_labels_df,
                              n_slices=N_SLICES, max_series=MAX_SERIES)
gold_loader = torch.utils.data.DataLoader(gold_ds, batch_size=BATCH_SIZE)

per_fold_gold = []
for fold in range(N_FOLDS):
    # pretrained=False: every weight is overwritten by the checkpoint on the next
    # line, so downloading ImageNet weights here only risks a hang
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=False).to(device)
    model.load_state_dict(torch.load(
        f'/kaggle/working/knee_phase5b_fold{fold}.pt', weights_only=True))
    _, yp = evaluate(model, gold_loader, device=device)
    per_fold_gold.append(yp)
    del model; gc.collect(); torch.cuda.empty_cache()

y_gold_true = gold_labels_df[LABEL_COLUMNS].to_numpy(dtype=float)

def gold_report(name, preds):
    auc = dict(zip(LABEL_COLUMNS, per_label_auc(y_gold_true, preds)))
    macro = float(np.nanmean(list(auc.values())))
    print(f'\ngold tier -- {name}: macro AUC {macro:.4f}')
    for k, v in auc.items():
        print(f'  {k:20s} {v:.4f}')
    return macro

macros = {}
for fold, yp in enumerate(per_fold_gold):
    macros[f'fold{fold}'] = gold_report(f'fold {fold} model', yp)
ensemble = np.mean(per_fold_gold, axis=0)
macros['ensemble5'] = gold_report('mean of 5 fold models', ensemble)
np.save('/kaggle/working/gold_pred_ensemble.npy', ensemble)

print('\n=== SUMMARY ===')
print(f'OOF vs pseudo-labels: {pooled_macro:.4f}')
for k, v in macros.items():
    print(f'gold transfer {k}: {v:.4f}')
ref = macros['ensemble5']
# The hypothesis is per-label, so read it per-label. Run 1's gold-transfer
# ensemble numbers, for the two labels this experiment exists to move:
RUN1_GOLD = {'MCL': 0.5034, 'Lateral OA': 0.6132, 'Medial OA': 0.7922,
             'Medial Meniscus': 0.6298, 'Lateral Meniscus': 0.7081,
             'ACL': 0.8051, 'Effusion': 0.9391}
print('\n=== hypothesis check: coronal labels vs run 1 (gold, ensemble of 5) ===')
_ens = dict(zip(LABEL_COLUMNS, per_label_auc(y_gold_true, ensemble)))
for k, before in RUN1_GOLD.items():
    print(f'  {k:20s} {before:.4f} -> {_ens[k]:.4f}  ({_ens[k] - before:+.4f})')
print(f'\ngold macro: 0.7252 (run 1) -> {macros["ensemble5"]:.4f} '
      f'({macros["ensemble5"] - 0.7252:+.4f}) -- n=58, directional only')
